# Scripts to generate plots for Figure 1

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

# --- 1. Load Data ---
df = pd.read_csv('/Users/laurabreimann/Documents/Postdoc/Colabs/Compression/compression_metrics.csv')

# --- 2. Color Scheme ---
COLORS = {
    'sparz': '#3367B0',    # Blue - SPARZ-Hybrid
    'h264': '#568259',     # Green - Lossy codecs
    'prores': '#568259',   
    'av1': '#568259',      
    'x265': '#568259',     
    'ffv1': '#EF8354',     # Orange - Lossless codecs
    'zstd': '#EF8354',     
    'gzip': '#BFC0C0',     # Silver - Control
}

ORDER = ['sparz', 'h264', 'prores', 'av1', 'x265', 'ffv1', 'zstd', 'gzip']
LABELS = ['SPARZ-Hybrid', 'h264', 'ProRes', 'AV1', 'x265', 'FFV1', 'ZSTD', 'gzip']

# Reorder dataframe 
df = df.set_index('codec').loc[ORDER].reset_index()


# --- 3. Plotting Function  ---
def create_dot_plot(ax, values, ylabel, ylim=None):
    colors = [COLORS[c] for c in ORDER]
    x = np.arange(len(ORDER))
    
    # Draw horizontal grid lines 
    ax.yaxis.grid(True, linestyle='--', alpha=0.4, color='gray', zorder=0)
    ax.set_axisbelow(True)
    
    # Plot the dots
    ax.scatter(x, values, s=140, c=colors, edgecolor='black', linewidth=0.5, zorder=3)
    
    ax.set_xticks(x)
    ax.set_xticklabels(LABELS, rotation=45, ha='right', fontweight='bold')
    ax.set_ylabel(ylabel, fontweight='bold', fontsize=12)

    
    if ylim:
        ax.set_ylim(ylim)
    
    # Add vertical separator lines between the categories
    ax.axvline(x=0.5, color='black', linestyle='-', linewidth=1, alpha=0.3)
    ax.axvline(x=4.5, color='black', linestyle='-', linewidth=1, alpha=0.3)
    ax.axvline(x=6.5, color='black', linestyle='-', linewidth=1, alpha=0.3)
    

    # Clean spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['left'].set_linewidth(1.5)

# --- 4. Generate the Figure ---
plt.rcParams['font.family'] = 'Arial'

# Create a 4x1 grid 
fig, axes = plt.subplots(4, 1, figsize=(7, 16), dpi=300)

create_dot_plot(axes[0], df['file_size_pct'].values, 'File size (%)', ylim=(0, 105))
create_dot_plot(axes[1], df['median_ssim'].values, 'SSIM',  ylim=(0.85, 1.02))
create_dot_plot(axes[2], df['jaccard'].values, 'Jaccard index',  ylim=(0, 1.05))
create_dot_plot(axes[3], df['l2_error'].values, 'L2 density error', ylim=(-0.002, 0.08))

plt.tight_layout(h_pad=3.0) 



# Save as PDF 
output_pdf = '/Users/laurabreimann/Documents/Postdoc/Colabs/Compression/Figures/figure1_plots/Figure1_Panels_B-E.pdf'
plt.savefig(output_pdf, format='pdf', transparent=True)
print(f"Saved PDF to: {output_pdf}")

plt.show()

Saved PDF to: /Users/laurabreimann/Documents/Postdoc/Colabs/Compression/Figures/figure1_plots/Figure1_Panels_B-E.pdf


/var/folders/n4/82zt36bj3190s97jwk9jxc580000gn/T/ipykernel_95470/3907897846.py:88: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [3]:
#!/usr/bin/env python3
"""
Create supplementary figure comparing SPARZ with traditional compression methods.
"""

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from pathlib import Path

# Output directory
OUTPUT_DIR = Path("/Users/laurabreimann/Documents/Postdoc/Colabs/Compression/Figures/figure1_plots")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set style
sns.set_style("white")

# Original file size for percentage calculation
original_size_mb = 41.65

# Data from all methods (sorted by file size)
data = {
    'Method': [
        'SPARZ-Hybrid', 'bzip2', 'Blosc', 'xz/lzma', 'Bitshuffle', 'JPEG2000',
        'ZSTD', 'gzip', 'HDF5', 'TIFF-DEFLATE', 'PNG', 'Zarr', 'FFV1', 'LZ4'
    ],
    'File size (%)': [
        49.1, 56.4, 57.3, 58.8, 60.3, 60.6,
        63.1, 70.6, 70.7, 73.6, 75.2, 77.4, 79.9, 86.4
    ],
    'Type': [
        'SPARZ-Hybrid', 'Traditional Lossless', 'Traditional Lossless',
        'Traditional Lossless', 'Traditional Lossless', 'Traditional Lossless',
        'SPARZ-Lossless', 'Traditional Lossless',
        'Traditional Lossless', 'Traditional Lossless', 'Traditional Lossless',
        'Traditional Lossless', 'SPARZ-Lossless', 'Traditional Lossless'
    ],
    'Lossless': [
        'Near-lossless', 'Lossless', 'Lossless', 'Lossless', 'Lossless',
        'Lossless', 'Lossless', 'Lossless',
        'Lossless', 'Lossless', 'Lossless', 'Lossless', 'Lossless', 'Lossless'
    ]
}

df = pd.DataFrame(data)

# Sort by file size
df = df.sort_values('File size (%)')

# Color palette
colors = {
    'SPARZ-Hybrid': '#3367B0',            # Blue
    'SPARZ-Lossless': '#EF8354',        # Orange
    'Traditional Lossless': '#BFC0C0'     # Gray
}

# Create the figure
fig, ax = plt.subplots(figsize=(8, 6))

# Create horizontal bar chart
bars = ax.barh(
    df['Method'],
    df['File size (%)'],
    color=[colors[t] for t in df['Type']],
    edgecolor='black',
    linewidth=0.5
)
# set font to arial
plt.rcParams['font.family'] = 'Arial'

# Add value labels on bars
for bar, val in zip(bars, df['File size (%)']):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

# Styling
ax.set_xlabel('File size (% of original)', fontsize=11)
ax.set_xlim(0, 100)
ax.axvline(x=100, color='gray', linestyle='--', linewidth=1, alpha=0.5)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3367B0', edgecolor='black', label='SPARZ-Hybrid'),
    Patch(facecolor='#EF8354', edgecolor='black', label='SPARZ-Lossless'),
    Patch(facecolor='#BFC0C0', edgecolor='black', label='Traditional Lossless')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

# Remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'supplement_compression_comparison.pdf', dpi=300, bbox_inches='tight')
plt.savefig(OUTPUT_DIR / 'supplement_compression_comparison.png', dpi=300, bbox_inches='tight')
plt.close()

print("Saved supplement figure")

# Also create a table for the supplement
print("\n" + "=" * 80)
print("SUPPLEMENTARY TABLE: Compression Method Comparison")
print("=" * 80)
print(f"{'Method':<20} {'Type':<25} {'File size (%)':<15} {'SSIM':<10} {'Jaccard':<10}")
print("-" * 80)
for _, row in df.iterrows():
    ssim = '1.0000' if row['Lossless'] == 'Lossless' else '0.9987'
    jaccard = '1.0000' if row['Lossless'] == 'Lossless' else '0.8646'
    print(f"{row['Method']:<20} {row['Type']:<25} {row['File size (%)']:<15.1f} {ssim:<10} {jaccard:<10}")
print("=" * 80)
print("\nNote: All traditional methods are fully lossless (SSIM=1.0, Jaccard=1.0).")
print("SPARZ achieves better compression than all traditional lossless methods")
print("while maintaining near-lossless quality for SMLM analysis.")


Saved supplement figure

SUPPLEMENTARY TABLE: Compression Method Comparison
Method               Type                      File size (%)   SSIM       Jaccard   
--------------------------------------------------------------------------------
SPARZ-Hybrid         SPARZ-Hybrid              49.1            0.9987     0.8646    
bzip2                Traditional Lossless      56.4            1.0000     1.0000    
Blosc                Traditional Lossless      57.3            1.0000     1.0000    
xz/lzma              Traditional Lossless      58.8            1.0000     1.0000    
Bitshuffle           Traditional Lossless      60.3            1.0000     1.0000    
JPEG2000             Traditional Lossless      60.6            1.0000     1.0000    
ZSTD                 SPARZ-Lossless            63.1            1.0000     1.0000    
gzip                 Traditional Lossless      70.6            1.0000     1.0000    
HDF5                 Traditional Lossless      70.7            1.0000     1.00